# IS3107 Chess ML Model

This notebook is organized as a simple pipeline:

1. Install dependencies
2. Configure the run
3. Define helper functions
4. Download and parse Lichess PGN data
5. Inspect missing evaluation coverage
6. Fill missing evaluations with Stockfish
7. Inspect the filled dataset before feature engineering

Notes:

- Existing Lichess `%eval` annotations are preserved.
- Stockfish is only used for rows where both `eval_cp` and `mate_in` are missing.
- Full-engine scoring on every missing row can be expensive. Start with smaller settings, validate, then scale up.
        


In [13]:
!pip install requests zstandard python-chess pandas


In [71]:
# Run configuration
BROADCAST_URLS = [
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2026-03.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2026-02.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2026-01.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2025-12.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2025-11.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2025-10.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2025-09.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2025-08.pgn.zst"
    
]

# Filter config
TARGET_FILTERED_GAMES = 2000 # not exactly 2000 games, estimated games are 2000. Due to lesser black win games, 
                            #min black win game count shld be satisfied which could result 2000+/-. But most of the time it will be 2000
# Optional minority-class import guard: keep reading until at least this many Black-win games are collected.
MIN_BLACK_WIN_GAMES = 200  # previosuly black win games were way lesser than white win games.
# Optional caps for majority classes so extra imports do not keep overfilling White wins and draws.
MAX_WHITE_WIN_GAMES = 900
MAX_DRAW_GAMES = 900
MIN_ELO = 2500
MAX_GAME_AGE_DAYS = 730
MIN_TIME_CONTROL_SECONDS = 15 * 60

# Engine-eval config
FILL_MISSING_EVALS = True
STOCKFISH_PATH = None  # Set this manually if stockfish is not on PATH.
ENGINE_DEPTH = 1       # Use shallow depth for scalable feature generation.
ENGINE_TIME_LIMIT = 0.01
ENGINE_MAX_ROWS = None  # Set to an int for a quick test run.
        


In [73]:
import io
import os
import re
import shutil
from typing import Dict, List, Optional

import chess
import chess.engine
import chess.pgn
import pandas as pd
import requests
import zstandard as zstd
from tqdm import tqdm


DEFAULT_STOCKFISH_CANDIDATES = [
    STOCKFISH_PATH,
    os.environ.get("STOCKFISH_PATH"),
    shutil.which("stockfish"),
    "/opt/homebrew/bin/stockfish",
    "/usr/local/bin/stockfish",
]


def stream_text_from_zst_url(url: str) -> io.TextIOBase:
    resp = requests.get(url, stream=True, timeout=120)
    resp.raise_for_status()

    dctx = zstd.ZstdDecompressor()
    reader = dctx.stream_reader(resp.raw)
    return io.TextIOWrapper(reader, encoding="utf-8", errors="replace")


def parse_clock_to_seconds(clock_str: str) -> Optional[float]:
    parts = clock_str.split(":")
    try:
        parts = [float(x) for x in parts]
    except ValueError:
        return None

    if len(parts) == 3:
        h, m, s = parts
        return h * 3600 + m * 60 + s
    if len(parts) == 2:
        m, s = parts
        return m * 60 + s
    if len(parts) == 1:
        return parts[0]
    return None


def parse_lichess_date(date_str: Optional[str]) -> Optional[pd.Timestamp]:
    if not date_str or date_str in {"????.??.??", "????.??.??"}:
        return None
    try:
        return pd.to_datetime(date_str, format="%Y.%m.%d", errors="raise").normalize()
    except (TypeError, ValueError):
        return None


def parse_time_control_base_seconds(time_control: Optional[str]) -> Optional[int]:
    if not time_control or time_control in {"-", "?"}:
        return None

    base_str = time_control.split("+", 1)[0]
    if not base_str.isdigit():
        return None
    return int(base_str)


def game_matches_filters(
    headers,
    min_elo: int,
    min_time_control_seconds: int,
    earliest_date: pd.Timestamp,
    latest_date: pd.Timestamp,
) -> bool:
    white_elo_raw = headers.get("WhiteElo")
    black_elo_raw = headers.get("BlackElo")
    white_elo = int(white_elo_raw) if white_elo_raw and white_elo_raw.isdigit() else None
    black_elo = int(black_elo_raw) if black_elo_raw and black_elo_raw.isdigit() else None
    has_super_gm = (white_elo is not None and white_elo >= min_elo) or (black_elo is not None and black_elo >= min_elo)

    game_date = parse_lichess_date(headers.get("Date"))
    if game_date is None:
        return False

    base_seconds = parse_time_control_base_seconds(headers.get("TimeControl"))
    if base_seconds is None:
        return False

    is_recent_enough = earliest_date <= game_date <= latest_date
    is_long_time_control = base_seconds >= min_time_control_seconds
    return has_super_gm and is_recent_enough and is_long_time_control


def extract_eval_and_clock(comment: str):
    eval_cp = None
    mate_in = None
    clk_seconds = None

    eval_match = re.search(r"\[%eval\s+([^\]]+)\]", comment)
    clk_match = re.search(r"\[%clk\s+([^\]]+)\]", comment)

    if eval_match:
        raw_eval = eval_match.group(1).strip()
        if raw_eval.startswith("#"):
            try:
                mate_in = int(raw_eval[1:])
            except ValueError:
                mate_in = None
        else:
            try:
                eval_cp = int(round(float(raw_eval) * 100))
            except ValueError:
                eval_cp = None

    if clk_match:
        clk_seconds = parse_clock_to_seconds(clk_match.group(1).strip())

    return eval_cp, mate_in, clk_seconds


def result_to_label(result: str) -> str:
    if result == "1-0":
        return "white_win"
    if result == "0-1":
        return "black_win"
    if result == "1/2-1/2":
        return "draw"
    return "unknown"


def find_stockfish_binary(stockfish_path: Optional[str] = None) -> str:
    candidates = [stockfish_path, *DEFAULT_STOCKFISH_CANDIDATES]
    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            return candidate

    raise FileNotFoundError(
        "Stockfish binary not found. Install Stockfish locally and set STOCKFISH_PATH if needed."
    )


def score_fen_with_stockfish(
    engine: chess.engine.SimpleEngine,
    fen: str,
    depth: int = 1,
    time_limit: Optional[float] = 0.01,
):
    board = chess.Board(fen)
    limit = chess.engine.Limit(depth=depth) if time_limit is None else chess.engine.Limit(depth=depth, time=time_limit)
    info = engine.analyse(board, limit)
    score = info["score"].white()

    if score.is_mate():
        return None, score.mate()
    return score.score(mate_score=100000), None


def fill_missing_evals_with_stockfish(
    df: pd.DataFrame,
    stockfish_path: Optional[str] = None,
    depth: int = 1,
    time_limit: Optional[float] = 0.01,
    max_rows: Optional[int] = None,
) -> pd.DataFrame:
    missing_mask = df["eval_cp"].isna() & df["mate_in"].isna()
    missing_idx = list(df.index[missing_mask])

    if max_rows is not None:
        missing_idx = missing_idx[:max_rows]

    if not missing_idx:
        print("No missing evals to fill.")
        return df

    engine_path = find_stockfish_binary(stockfish_path)
    print(f"Using Stockfish at: {engine_path}")
    print(f"Rows queued for engine eval: {len(missing_idx)}")

    filled_df = df.copy()
    fen_cache = {}

    with chess.engine.SimpleEngine.popen_uci(engine_path) as engine:
        for idx in tqdm(missing_idx, desc="Computing missing evals", unit="row"):
            fen = filled_df.at[idx, "fen_after"]
            if fen not in fen_cache:
                fen_cache[fen] = score_fen_with_stockfish(
                    engine,
                    fen,
                    depth=depth,
                    time_limit=time_limit,
                )
            eval_cp, mate_in = fen_cache[fen]
            filled_df.at[idx, "eval_cp"] = eval_cp
            filled_df.at[idx, "mate_in"] = mate_in

    return filled_df


def load_filtered_broadcast_games(
    target_games: int = 1000,
    min_black_win_games: Optional[int] = None,
    max_white_win_games: Optional[int] = None,
    max_draw_games: Optional[int] = None,
    min_elo: int = 2700,
    max_game_age_days: int = 730,
    min_time_control_seconds: int = 15 * 60,
    urls: Optional[List[str]] = None,
) -> pd.DataFrame:
    if urls is None:
        urls = BROADCAST_URLS

    rows: List[Dict] = []
    today = pd.Timestamp.today().normalize()
    earliest_date = today - pd.Timedelta(days=max_game_age_days)
    matched_games = 0
    # Track accepted game counts by outcome so import can cap majority classes and monitor Black wins.
    accepted_game_counts = {"white_win": 0, "draw": 0, "black_win": 0}

    with tqdm(total=target_games, desc="Parsing filtered games", unit="game") as pbar:
        for url in urls:
            # Stop once the overall game target is met and the optional Black-win target is also satisfied.
            if matched_games >= target_games and (
                min_black_win_games is None or accepted_game_counts["black_win"] >= min_black_win_games
            ):
                break

            print(f"Reading from: {url}")
            text_stream = stream_text_from_zst_url(url)

            while True:
                # Keep reading past `target_games` only when extra Black-win games are explicitly requested.
                if matched_games >= target_games and (
                    min_black_win_games is None or accepted_game_counts["black_win"] >= min_black_win_games
                ):
                    break

                game = chess.pgn.read_game(text_stream)
                if game is None:
                    break

                headers = game.headers
                if not game_matches_filters(
                    headers,
                    min_elo=min_elo,
                    min_time_control_seconds=min_time_control_seconds,
                    earliest_date=earliest_date,
                    latest_date=today,
                ):
                    continue

                board = game.board()

                result = headers.get("Result", "")
                # Reuse the same per-game label everywhere so class-balance tracking stays consistent.
                game_label = result_to_label(result)

                # Skip games with unsupported or missing final results.
                if game_label == "unknown":
                    continue

                # Skip extra majority-class games once their optional caps are reached.
                if game_label == "white_win" and max_white_win_games is not None and accepted_game_counts["white_win"] >= max_white_win_games:
                    continue
                if game_label == "draw" and max_draw_games is not None and accepted_game_counts["draw"] >= max_draw_games:
                    continue
                white_elo = headers.get("WhiteElo")
                black_elo = headers.get("BlackElo")
                site = headers.get("Site", "")
                game_id = site.rstrip("/").split("/")[-1] if site else None

                node = game
                ply = 0

                while node.variations:
                    next_node = node.variation(0)
                    move = next_node.move
                    ply += 1

                    fen_before = board.fen()
                    side_to_move = "white" if board.turn == chess.WHITE else "black"
                    san = board.san(move)
                    uci = move.uci()

                    board.push(move)
                    fen_after = board.fen()

                    comment = next_node.comment or ""
                    eval_cp, mate_in, clk_seconds = extract_eval_and_clock(comment)

                    rows.append(
                        {
                            "game_id": game_id,
                            "date": headers.get("Date"),
                            "white_player": headers.get("White"),
                            "black_player": headers.get("Black"),
                            "white_elo": int(white_elo) if white_elo and white_elo.isdigit() else None,
                            "black_elo": int(black_elo) if black_elo and black_elo.isdigit() else None,
                            "result": result,
                            "label": game_label,
                            "time_control": headers.get("TimeControl"),
                            "eco": headers.get("ECO"),
                            "opening": headers.get("Opening"),
                            "ply": ply,
                            "side_to_move": side_to_move,
                            "san": san,
                            "uci": uci,
                            "fen_before": fen_before,
                            "fen_after": fen_after,
                            "eval_cp": eval_cp,
                            "mate_in": mate_in,
                            "clock_seconds_after_move": clk_seconds,
                        }
                    )

                    node = next_node

                matched_games += 1
                # Update accepted game counts after this game is kept.
                accepted_game_counts[game_label] += 1
                pbar.update(1)

    return pd.DataFrame(rows)
        


In [75]:
# Step 1: Download and parse the dataset
df = load_filtered_broadcast_games(
    target_games=TARGET_FILTERED_GAMES,
    min_black_win_games=MIN_BLACK_WIN_GAMES,
    max_white_win_games=MAX_WHITE_WIN_GAMES,
    max_draw_games=MAX_DRAW_GAMES,
    min_elo=MIN_ELO,
    max_game_age_days=MAX_GAME_AGE_DAYS,
    min_time_control_seconds=MIN_TIME_CONTROL_SECONDS,
)

# Summarize outcome balance at the game level so class imbalance is visible immediately after import.
game_outcome_counts = (
    df.groupby("game_id")["label"]
    .first()
    .value_counts()
    .reindex(["white_win", "draw", "black_win"], fill_value=0)
)
# Convert game counts into proportions for presentation-friendly reporting.
game_outcome_proportions = (game_outcome_counts / game_outcome_counts.sum()).round(4)

print(df.head())
print("Rows:", len(df))
print("Unique games:", df["game_id"].nunique())
print("Estimated rows target:", TARGET_FILTERED_GAMES * 200)
print("Game-level outcome counts:")
print(game_outcome_counts)
print("Game-level outcome proportions:")
print(game_outcome_proportions)
        


Parsing filtered games:   0%|                                                                          | 0/2000 [00:00<?, ?game/s]

Reading from: https://database.lichess.org/broadcast/lichess_db_broadcast_2026-03.pgn.zst


Parsing filtered games:  12%|███████▋                                                        | 241/2000 [00:47<05:53,  4.97game/s]

Reading from: https://database.lichess.org/broadcast/lichess_db_broadcast_2026-02.pgn.zst


Parsing filtered games:  43%|███████████████████████████▎                                    | 853/2000 [01:45<02:16,  8.42game/s]

...

Reading from: https://database.lichess.org/broadcast/lichess_db_broadcast_2025-11.pgn.zst


illegal san: '0-0' in rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 while parsing <Game at 0x34e11d970 ('Martinez Gonzalez, Miguel A.' vs. 'Ramirez Maillo, Cristian', '2025.11.02' at 'https://lichess.org/broadcast/benidorm-chess-open-2025--group-a/round-9/zqiflA9E/ptwjkSg1')>
Parsing filtered games:  96%|████████████████████████████████████████████████████████████▋  | 1925/2000 [04:54<04:14,  3.40s/game]

Reading from: https://database.lichess.org/broadcast/lichess_db_broadcast_2025-10.pgn.zst


illegal san: 'Nbc6' in rbnkqnbr/pppppppp/8/8/8/5P2/PPPPP1PP/RBNKQNBR b KQkq - 0 1 while parsing <Game at 0x34498d040 ('Ethereal-FRC 14.40-frc' vs. 'Caissa 1.23', '2025.10.09' at 'https://lichess.org/broadcast/tcec-double-fischer-random-chess-4--league-d/round-1/RR2B3OXh/yJt7082W')>
illegal san: 'Nbc6' in rbnkqnbr/pppppppp/8/8/8/5P2/PPPPP1PP/RBNKQNBR b KQkq - 0 1 while parsing <Game at 0x32e148dd0 ('Caissa 1.23' vs. 'Ethereal-FRC 14.40-frc', '2025.10.09' at 'https://lichess.org/broadcast/tcec-double-fischer-random-chess-4--league-d/round-1/RR2B3OXh/W4aKKU0D')>
Parsing filtered games: 100%|███████████████████████████████████████████████████████████████| 2000/2000 [05:41<00:00,  5.86game/s]


    game_id        date      white_player        black_player  white_elo  \
0  wjVeV6zD  2026.01.20  Berserk 20250622  Obsidian dev-16.15       3667   
1  wjVeV6zD  2026.01.20  Berserk 20250622  Obsidian dev-16.15       3667   
2  wjVeV6zD  2026.01.20  Berserk 20250622  Obsidian dev-16.15       3667   
3  wjVeV6zD  2026.01.20  Berserk 20250622  Obsidian dev-16.15       3667   
4  wjVeV6zD  2026.01.20  Berserk 20250622  Obsidian dev-16.15       3667   

   black_elo   result label time_control  eco  \
0     3714.0  1/2-1/2  draw       3600+6  B27   
1     3714.0  1/2-1/2  draw       3600+6  B27   
2     3714.0  1/2-1/2  draw       3600+6  B27   
3     3714.0  1/2-1/2  draw       3600+6  B27   
4     3714.0  1/2-1/2  draw       3600+6  B27   

                                     opening  ply side_to_move  san   uci  \
0  Sicilian Defense: Hyperaccelerated Dragon    1        white  Nf3  g1f3   
1  Sicilian Defense: Hyperaccelerated Dragon    2        black   g6  g7g6   
2  Sicilian Defen

## Step 5: Feature Engineering
We keep feature engineering explicit and interpretable. Missing `eval_cp` values are filled first, then the remaining modeling features are created one by one.


### 5.1 Fill Missing `eval_cp`


In [77]:
import numpy as np

model_df = df.copy()

total_rows = len(model_df)
eval_cp_missing_before = model_df["eval_cp"].isna().sum()
mate_in_non_null_before = model_df["mate_in"].notna().sum()
rows_with_any_eval_before = (model_df["eval_cp"].notna() | model_df["mate_in"].notna()).sum()
rows_without_any_eval_before = total_rows - rows_with_any_eval_before

if FILL_MISSING_EVALS:
    model_df = fill_missing_evals_with_stockfish(
        model_df,
        stockfish_path=STOCKFISH_PATH,
        depth=ENGINE_DEPTH,
        time_limit=ENGINE_TIME_LIMIT,
        max_rows=ENGINE_MAX_ROWS,
    )
else:
    print("Skipping Stockfish fill.")



Using Stockfish at: /opt/homebrew/bin/stockfish
Rows queued for engine eval: 51356


Computing missing evals: 100%|███████████████████████████████████████████████████████████| 51356/51356 [00:39<00:00, 1314.45row/s]


### Validate `eval_cp` Fill


In [79]:
# `eval_cp` and `mate_in` both come from the engine's evaluation of the current position.
# If the position is still being judged in centipawns, the signal appears in `eval_cp`.
# If the engine sees a forced checkmate sequence, it may report that as `mate_in` instead.
# In that sense, `mate_in` is not a completely separate concept from `eval_cp`; it is a stronger,
# more decisive version of engine evaluation for positions where mate is already forced.
# For validation, we therefore check whether a row has either `eval_cp` or `mate_in`, because
# either one means the row contains some engine-based signal about the position.

In [80]:
eval_cp_missing_after = model_df["eval_cp"].isna().sum()
mate_in_non_null_after = model_df["mate_in"].notna().sum()
rows_with_any_eval_after = (model_df["eval_cp"].notna() | model_df["mate_in"].notna()).sum()
rows_without_any_eval_after = total_rows - rows_with_any_eval_after
filled_eval_rows = eval_cp_missing_before - eval_cp_missing_after

print("=== eval_cp fill validation ===")
print("Total rows:", total_rows)
print("Rows without eval_cp and mate_in before fill:", rows_without_any_eval_before)
print("Rows without eval_cp and mate_in after fill:", rows_without_any_eval_after)
print("Rows without eval_cp before fill:", eval_cp_missing_before)
print("Rows without eval_cp after fill:", eval_cp_missing_after)
print("New eval_cp values filled:", filled_eval_rows)
print("Rows with mate_in before fill:", mate_in_non_null_before)
print("Rows with mate_in after fill:", mate_in_non_null_after)
print("Rows with any eval signal before fill:", rows_with_any_eval_before)
print("Rows with any eval signal after fill:", rows_with_any_eval_after)
print("Fraction with any eval signal after fill:", rows_with_any_eval_after / total_rows if total_rows else 0)

eval_series = model_df["eval_cp"].dropna()
print("=== eval_cp stats after fill ===")
print("min:", eval_series.min())
print("max:", eval_series.max())
print("positive:", (eval_series > 0).sum())
print("negative:", (eval_series < 0).sum())
print("zero:", (eval_series == 0).sum())

model_df[model_df["eval_cp"].notna() | model_df["mate_in"].notna()].head(10)



=== eval_cp fill validation ===
Total rows: 322192
Rows without eval_cp and mate_in before fill: 51356
Rows without eval_cp and mate_in after fill: 0
Rows without eval_cp before fill: 72309
Rows without eval_cp after fill: 22066
New eval_cp values filled: 50243
Rows with mate_in before fill: 20953
Rows with mate_in after fill: 22066
Rows with any eval signal before fill: 270836
Rows with any eval signal after fill: 322192
Fraction with any eval signal after fill: 1.0
=== eval_cp stats after fill ===
min: -8343.0
max: 19908.0
positive: 248681
negative: 31005
zero: 20440


,game_id,date,white_player,black_player,white_elo,black_elo,result,label,time_control,eco,opening,ply,side_to_move,san,uci,fen_before,fen_after,eval_cp,mate_in,clock_seconds_after_move
0,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714.0,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,1,white,Nf3,g1f3,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,rnbqkbnr/pppppppp/8/8/8/5N2/PPPPPPPP/RNBQKB1R ...,10.0,NaN,3600.0
1,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714.0,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,2,black,g6,g7g6,rnbqkbnr/pppppppp/8/8/8/5N2/PPPPPPPP/RNBQKB1R ...,rnbqkbnr/pppppp1p/6p1/8/8/5N2/PPPPPPPP/RNBQKB1...,38.0,NaN,3600.0
2,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714.0,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,3,white,e4,e2e4,rnbqkbnr/pppppp1p/6p1/8/8/5N2/PPPPPPPP/RNBQKB1...,rnbqkbnr/pppppp1p/6p1/8/4P3/5N2/PPPP1PPP/RNBQK...,51.0,NaN,3600.0
3,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714.0,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,4,black,c5,c7c5,rnbqkbnr/pppppp1p/6p1/8/4P3/5N2/PPPP1PPP/RNBQK...,rnbqkbnr/pp1ppp1p/6p1/2p5/4P3/5N2/PPPP1PPP/RNB...,34.0,NaN,3600.0
4,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714.0,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,5,white,c3,c2c3,rnbqkbnr/pp1ppp1p/6p1/2p5/4P3/5N2/PPPP1PPP/RNB...,rnbqkbnr/pp1ppp1p/6p1/2p5/4P3/2P2N2/PP1P1PPP/R...,24.0,NaN,3600.0
5,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714.0,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,6,black,Nf6,g8f6,rnbqkbnr/pp1ppp1p/6p1/2p5/4P3/2P2N2/PP1P1PPP/R...,rnbqkb1r/pp1ppp1p/5np1/2p5/4P3/2P2N2/PP1P1PPP/...,49.0,NaN,3600.0
6,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714.0,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,7,white,e5,e4e5,rnbqkb1r/pp1ppp1p/5np1/2p5/4P3/2P2N2/PP1P1PPP/...,rnbqkb1r/pp1ppp1p/5np1/2p1P3/8/2P2N2/PP1P1PPP/...,48.0,NaN,3600.0
7,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714.0,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,8,black,Nd5,f6d5,rnbqkb1r/pp1ppp1p/5np1/2p1P3/8/2P2N2/PP1P1PPP/...,rnbqkb1r/pp1ppp1p/6p1/2pnP3/8/2P2N2/PP1P1PPP/R...,53.0,NaN,3600.0
8,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714.0,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,9,white,d4,d2d4,rnbqkb1r/pp1ppp1p/6p1/2pnP3/8/2P2N2/PP1P1PPP/R...,rnbqkb1r/pp1ppp1p/6p1/2pnP3/3P4/2P2N2/PP3PPP/R...,47.0,NaN,3600.0
9,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714.0,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,10,black,cxd4,c5d4,rnbqkb1r/pp1ppp1p/6p1/2pnP3/3P4/2P2N2/PP3PPP/R...,rnbqkb1r/pp1ppp1p/6p1/3nP3/3p4/2P2N2/PP3PPP/RN...,44.0,NaN,3600.0


### 5.2 Elo Difference Feature


In [82]:
# Elo difference captures the overall skill gap between White and Black.
model_df["elo_diff"] = model_df["white_elo"] - model_df["black_elo"]

model_df[["white_elo", "black_elo", "elo_diff"]].head(10)



,white_elo,black_elo,elo_diff
0,3667,3714.0,-47.0
1,3667,3714.0,-47.0
2,3667,3714.0,-47.0
3,3667,3714.0,-47.0
4,3667,3714.0,-47.0
5,3667,3714.0,-47.0
6,3667,3714.0,-47.0
7,3667,3714.0,-47.0
8,3667,3714.0,-47.0
9,3667,3714.0,-47.0


### 5.3 Time Left Feature


In [84]:
# Remaining clock time captures time pressure after the move was played.
model_df["time_left_seconds"] = model_df["clock_seconds_after_move"].clip(lower=0)

model_df[["clock_seconds_after_move", "time_left_seconds"]].head(10)



,clock_seconds_after_move,time_left_seconds
0,3600.0,3600.0
1,3600.0,3600.0
2,3600.0,3600.0
3,3600.0,3600.0
4,3600.0,3600.0
5,3600.0,3600.0
6,3600.0,3600.0
7,3600.0,3600.0
8,3600.0,3600.0
9,3600.0,3600.0


### 5.4 Position Context Features


In [86]:
# Engine evaluation is the strongest signal of which side is better in the current position.
model_df["eval_cp_clipped"] = model_df["eval_cp"].clip(-1000, 1000)
# Ply tells the model how early or late the position is in the game.
model_df["ply"] = model_df["ply"].astype(int)
# Side to move tells the model whose turn it is in the current position.
model_df["side_to_move"] = model_df["side_to_move"].astype("category")

model_df[["eval_cp", "eval_cp_clipped", "ply", "side_to_move"]].head(10)



,eval_cp,eval_cp_clipped,ply,side_to_move
0,10.0,10.0,1,white
1,38.0,38.0,2,black
2,51.0,51.0,3,white
3,34.0,34.0,4,black
4,24.0,24.0,5,white
5,49.0,49.0,6,black
6,48.0,48.0,7,white
7,53.0,53.0,8,black
8,47.0,47.0,9,white
9,44.0,44.0,10,black


### 5.5 Final Modeling Table


In [88]:
required_columns = [
    "game_id",
    "label",
    "eval_cp_clipped",
    "white_elo",
    "black_elo",
    "elo_diff",
    "time_left_seconds",
    "ply",
    "side_to_move",
]

modeling_input_df = model_df.copy()

model_df = model_df.dropna(subset=required_columns).copy()

feature_cols = [
    "eval_cp_clipped",
    "elo_diff",
    "time_left_seconds",
    "ply",
    "side_to_move",
]

label_order = ["white_win", "draw", "black_win"]
label_to_id = {label: idx for idx, label in enumerate(label_order)}
model_df = model_df[model_df["label"].isin(label_to_id)].copy()
model_df["target"] = model_df["label"].map(label_to_id).astype(int)

print("Rows kept for modeling:", len(model_df))
print("Unique games kept:", model_df["game_id"].nunique())
print("Feature columns:", feature_cols)
print(model_df[["label", "target"]].drop_duplicates().sort_values("target"))



Rows kept for modeling: 299150
Unique games kept: 1998
Feature columns: ['eval_cp_clipped', 'elo_diff', 'time_left_seconds', 'ply', 'side_to_move']
           label  target
122    white_win       0
0           draw       1
51615  black_win       2


## Step 6: Train / Validation / Test Split
We split by `game_id` instead of by row so that move rows from the same game do not leak across datasets.


In [91]:
from sklearn.model_selection import train_test_split

game_level_df = model_df.groupby("game_id", as_index=False).agg(game_label=("label", "first"))

train_val_games, test_games = train_test_split(
    game_level_df,
    test_size=0.15,
    random_state=42,
    stratify=game_level_df["game_label"],
)

val_size_within_train_val = 0.15 / 0.85
train_games, val_games = train_test_split(
    train_val_games,
    test_size=val_size_within_train_val,
    random_state=42,
    stratify=train_val_games["game_label"],
)

train_ids = set(train_games["game_id"])
val_ids = set(val_games["game_id"])
test_ids = set(test_games["game_id"])

train_df = model_df[model_df["game_id"].isin(train_ids)].copy()
val_df = model_df[model_df["game_id"].isin(val_ids)].copy()
test_df = model_df[model_df["game_id"].isin(test_ids)].copy()

print("Train rows:", len(train_df), "| games:", train_df["game_id"].nunique())
print("Val rows:", len(val_df), "| games:", val_df["game_id"].nunique())
print("Test rows:", len(test_df), "| games:", test_df["game_id"].nunique())

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)


Train rows: 210850 | games: 1398
Val rows: 43866 | games: 300
Test rows: 44434 | games: 300


## Step 7: Train a LightGBM Multiclass Model
The model predicts three probabilities per move: `white_win_prob`, `draw_prob`, and `black_win_prob`.


In [93]:
from sklearn.metrics import accuracy_score, classification_report, log_loss

try:
    import lightgbm as lgb
except ImportError as exc:
    raise ImportError("LightGBM is required for this section. Run `pip install lightgbm`.") from exc

X_train = train_df[feature_cols].copy()
y_train = train_df["target"].copy()
X_val = val_df[feature_cols].copy()
y_val = val_df["target"].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df["target"].copy()

categorical_features = ["side_to_move"]

lgbm_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(label_order),
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

lgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=25),
    ],
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001477 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1022
[LightGBM] [Info] Number of data points in the train set: 210850, number of used features: 5
[LightGBM] [Info] Start training from score -0.768867
[LightGBM] [Info] Start training from score -0.810462
[LightGBM] [Info] Start training from score -2.388041
Training until validation scores don't improve for 30 rounds
[25]	valid_0's multi_logloss: 0.491529
[50]	valid_0's multi_logloss: 0.424816
[75]	valid_0's multi_logloss: 0.415299
[100]	valid_0's multi_logloss: 0.41245
[125]	valid_0's multi_logloss: 0.413663
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Early stopping, best iteration is:
[106]	valid_0's multi_logloss: 0.411811


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, max_depth=6,
               min_child_samples=50, n_estimators=300, num_class=3,
               objective='multiclass', random_state=42, subsample=0.8)

## Step 8: Evaluate on Validation and Test Sets


In [95]:
val_proba = lgbm_model.predict_proba(X_val)
test_proba = lgbm_model.predict_proba(X_test)

val_pred = val_proba.argmax(axis=1)
test_pred = test_proba.argmax(axis=1)

print('Validation log loss:', log_loss(y_val, val_proba, labels=[0, 1, 2]))
print('Validation accuracy:', accuracy_score(y_val, val_pred))
print('Test log loss:', log_loss(y_test, test_proba, labels=[0, 1, 2]))
print('Test accuracy:', accuracy_score(y_test, test_pred))

print('\nTest classification report:')
print(classification_report(y_test, test_pred, target_names=label_order))

feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgbm_model.feature_importances_,
}).sort_values('importance', ascending=False)

feature_importance_df


Validation log loss: 0.4118111799564892
Validation accuracy: 0.8260383896411799
Test log loss: 0.4241098552343067
Test accuracy: 0.8349462123599046

Test classification report:
              precision    recall  f1-score   support

   white_win       0.86      0.85      0.85     21154
        draw       0.81      0.85      0.83     18935
   black_win       0.84      0.69      0.76      4345

    accuracy                           0.83     44434
   macro avg       0.84      0.80      0.81     44434
weighted avg       0.84      0.83      0.83     44434



,feature,importance
1,elo_diff,4629
0,eval_cp_clipped,1883
3,ply,1444
2,time_left_seconds,1379
4,side_to_move,205


## Step 9: Generate Move-Level Outcome Probabilities
This adds three probabilities for each move row. For rows excluded from modeling because of missing values, probabilities remain `NaN`.


In [112]:
all_model_proba = lgbm_model.predict_proba(model_df[feature_cols])

df_with_probs = df.copy()
# Carry the clipped evaluation feature into the final probability dataframe for inspection.
df_with_probs['eval_cp_clipped'] = np.nan
df_with_probs['white_win_prob'] = np.nan
df_with_probs['draw_prob'] = np.nan
df_with_probs['black_win_prob'] = np.nan

df_with_probs.loc[model_df.index, 'eval_cp_clipped'] = model_df['eval_cp_clipped']
df_with_probs.loc[model_df.index, 'white_win_prob'] = all_model_proba[:, label_to_id['white_win']]
df_with_probs.loc[model_df.index, 'draw_prob'] = all_model_proba[:, label_to_id['draw']]
df_with_probs.loc[model_df.index, 'black_win_prob'] = all_model_proba[:, label_to_id['black_win']]

df_with_probs['prob_sum'] = df_with_probs[['white_win_prob', 'draw_prob', 'black_win_prob']].sum(axis=1)

display_cols = [
    'game_id',
    'ply',
    'side_to_move',
    'eval_cp',
    'white_elo',
    'black_elo',
    'clock_seconds_after_move',
    'label',
    'white_win_prob',
    'draw_prob',
    'black_win_prob',
    'prob_sum',
]

print('Rows with probabilities:', df_with_probs['white_win_prob'].notna().sum())
print('Probability sum min/max:', df_with_probs['prob_sum'].min(), df_with_probs['prob_sum'].max())
df_with_probs[display_cols].head(20)


Rows with probabilities: 299150
Probability sum min/max: 0.0 1.0000000000000002


,game_id,ply,side_to_move,eval_cp,white_elo,black_elo,clock_seconds_after_move,label,white_win_prob,draw_prob,black_win_prob,prob_sum
0,wjVeV6zD,1,white,10.0,3667,3714.0,3600.0,draw,0.264133,0.706836,0.029030,1.0
1,wjVeV6zD,2,black,38.0,3667,3714.0,3600.0,draw,0.276184,0.702260,0.021555,1.0
2,wjVeV6zD,3,white,51.0,3667,3714.0,3600.0,draw,0.270612,0.714539,0.014848,1.0
3,wjVeV6zD,4,black,34.0,3667,3714.0,3600.0,draw,0.276184,0.702260,0.021555,1.0
4,wjVeV6zD,5,white,24.0,3667,3714.0,3600.0,draw,0.279723,0.698840,0.021437,1.0
5,wjVeV6zD,6,black,49.0,3667,3714.0,3600.0,draw,0.270612,0.714539,0.014848,1.0
6,wjVeV6zD,7,white,48.0,3667,3714.0,3600.0,draw,0.270612,0.714539,0.014848,1.0
7,wjVeV6zD,8,black,53.0,3667,3714.0,3600.0,draw,0.268817,0.718163,0.013020,1.0
8,wjVeV6zD,9,white,47.0,3667,3714.0,3600.0,draw,0.270612,0.714539,0.014848,1.0
9,wjVeV6zD,10,black,44.0,3667,3714.0,3600.0,draw,0.270612,0.714539,0.014848,1.0


## Step 9A: Sanity Check the Predicted Probabilities
These quick views help confirm that the predicted probabilities change sensibly across different positions instead of only looking at the first 20 rows.


In [115]:
# Focus on rows where probabilities were successfully generated.
prob_check_df = df_with_probs[df_with_probs['white_win_prob'].notna()].copy()

# Reuse the same columns so the sanity checks stay easy to read in the notebook.
check_cols = [
    'game_id',
    'ply',
    'side_to_move',
    'eval_cp',
    'label',
    'white_win_prob',
    'draw_prob',
    'black_win_prob',
]

print('Random probability samples:')
display(prob_check_df[check_cols].sample(20, random_state=42))

# Check whether strongly White-favored engine positions produce higher White-win probabilities.
print('\nHighest eval_cp rows:')
display(prob_check_df.sort_values('eval_cp', ascending=False)[check_cols].head(10))

# Check whether strongly Black-favored engine positions produce higher Black-win probabilities.
print('\nLowest eval_cp rows:')
display(prob_check_df.sort_values('eval_cp', ascending=True)[check_cols].head(10))

# Summarize the overall probability distribution so one class is not silently dominating everything.
print('\nAverage predicted probabilities:')
print(prob_check_df[['white_win_prob', 'draw_prob', 'black_win_prob']].mean())


Random probability samples:


,game_id,ply,side_to_move,eval_cp,label,white_win_prob,draw_prob,black_win_prob
147821,3GyxuXvk,135,white,NaN,white_win,0.980282,0.018901,0.000818
194548,i32kqcs1,75,white,NaN,draw,0.043397,0.427948,0.528655
260144,PqIvPeik,72,black,145.0,white_win,0.862496,0.130210,0.007294
210410,xIx2ViAK,71,white,0.0,draw,0.010987,0.976536,0.012477
165420,Wb5Il5JK,78,black,NaN,white_win,0.172265,0.820560,0.007175
281400,vpjYwAkd,19,white,103.0,draw,0.360976,0.634893,0.004132
180568,oXopYfo8,130,black,497.0,white_win,0.995444,0.003619,0.000937
274046,mvS1FgWp,96,black,376.0,white_win,0.994555,0.004328,0.001116
231160,BEM0J9FI,39,white,-30.0,black_win,0.016301,0.035746,0.947953
72994,dfVGAU1D,132,black,485.0,white_win,0.988292,0.010776,0.000932



Highest eval_cp rows:


,game_id,ply,side_to_move,eval_cp,label,white_win_prob,draw_prob,black_win_prob
228954,oUnX0EY5,140,black,19908.0,white_win,0.997098,0.002525,0.000377
16610,mkFwSdNR,141,white,8368.0,white_win,0.995959,0.003424,0.000617
284434,Gsl9JbRN,187,white,8362.0,white_win,0.996717,0.002801,0.000482
284446,Gsl9JbRN,199,white,8360.0,white_win,0.996717,0.002801,0.000482
284442,Gsl9JbRN,195,white,8360.0,white_win,0.996717,0.002801,0.000482
50891,d4fizD9N,115,white,8360.0,white_win,0.994820,0.004479,0.000701
284441,Gsl9JbRN,194,black,8360.0,white_win,0.996046,0.003384,0.000569
284440,Gsl9JbRN,193,white,8360.0,white_win,0.996717,0.002801,0.000482
50892,d4fizD9N,116,black,8360.0,white_win,0.992066,0.007014,0.000920
75389,GUmkULb2,247,white,8358.0,white_win,0.992324,0.006937,0.000740



Lowest eval_cp rows:


,game_id,ply,side_to_move,eval_cp,label,white_win_prob,draw_prob,black_win_prob
310132,viRPZgzP,169,white,-8343.0,black_win,0.004625,0.011268,0.984106
318129,y47q7TYg,140,black,-8339.0,black_win,0.009958,0.026689,0.963352
310134,viRPZgzP,171,white,-8339.0,black_win,0.004750,0.011267,0.983983
313585,K7GVLtrR,128,black,-8330.0,black_win,0.030089,0.024320,0.945591
261663,9ZWsi3Ge,150,black,-8325.0,black_win,0.004817,0.012374,0.982809
317541,trr7cs1x,296,black,-8308.0,black_win,0.004246,0.015822,0.979933
244086,Sz6UX9Yb,296,black,-8308.0,black_win,0.711291,0.109312,0.179397
261671,9ZWsi3Ge,158,black,-8308.0,black_win,0.004790,0.017983,0.977227
317542,trr7cs1x,297,white,-8308.0,black_win,0.008569,0.027571,0.963860
309616,yNKsAFTX,189,white,-8308.0,black_win,0.006564,0.019007,0.974429



Average predicted probabilities:
white_win_prob    0.468425
draw_prob         0.440630
black_win_prob    0.090945
dtype: float64


## Step 9B: Inspect Rows with More Mixed Outcome Probabilities
This helps surface positions where the model assigns meaningful probability mass to all three outcomes, instead of only showing clearly winning or clearly drawn positions.


In [118]:
# Reuse the probability-check dataframe created above.
mixed_prob_cols = [
    'game_id',
    'ply',
    'side_to_move',
    'eval_cp',
    'eval_cp_clipped',
    'label',
    'white_win_prob',
    'draw_prob',
    'black_win_prob',
]

# Look for rows where all three probabilities stay in a non-trivial range.
mixed_prob_df = prob_check_df[
    prob_check_df['white_win_prob'].between(0.20, 0.60)
    & prob_check_df['draw_prob'].between(0.15, 0.60)
    & prob_check_df['black_win_prob'].between(0.10, 0.60)
].copy()

print('Rows with mixed outcome probabilities:', len(mixed_prob_df))
display(mixed_prob_df[mixed_prob_cols].head(20))

# Also show rows closest to a user-intuitive target mix such as 30/20/50.
target_mix = np.array([0.30, 0.20, 0.50])
prob_matrix = prob_check_df[['white_win_prob', 'draw_prob', 'black_win_prob']].to_numpy()
prob_check_df = prob_check_df.copy()
prob_check_df['target_mix_distance'] = ((prob_matrix - target_mix) ** 2).sum(axis=1) ** 0.5

print('\nRows closest to White 30% / Draw 20% / Black 50%:')
display(prob_check_df.sort_values('target_mix_distance')[mixed_prob_cols + ['target_mix_distance']].head(20))


Rows with mixed outcome probabilities: 1579


,game_id,ply,side_to_move,eval_cp,eval_cp_clipped,label,white_win_prob,draw_prob,black_win_prob
2352,UHqNk7s4,13,white,-29.0,-29.0,draw,0.229729,0.578412,0.191859
2354,UHqNk7s4,15,white,-28.0,-28.0,draw,0.214448,0.589309,0.196242
9087,W80gGT7y,6,black,-27.0,-27.0,draw,0.230988,0.599449,0.169563
17379,AdHdyb7Q,3,white,-9.0,-9.0,draw,0.295043,0.492761,0.212196
36848,PLY8uM2u,8,black,8.0,8.0,white_win,0.323717,0.575507,0.100775
36852,PLY8uM2u,12,black,9.0,9.0,white_win,0.323717,0.575507,0.100775
39745,uE1c6wlL,13,white,0.0,0.0,white_win,0.436998,0.415339,0.147663
39747,uE1c6wlL,15,white,1.0,1.0,white_win,0.443111,0.413980,0.142908
42104,XqcLOsA1,3,white,0.0,0.0,white_win,0.436998,0.415339,0.147663
42106,XqcLOsA1,5,white,0.0,0.0,white_win,0.436998,0.415339,0.147663



Rows closest to White 30% / Draw 20% / Black 50%:


,game_id,ply,side_to_move,eval_cp,eval_cp_clipped,label,white_win_prob,draw_prob,black_win_prob,target_mix_distance
232784,WQMXr9nD,44,black,1.0,1.0,white_win,0.293017,0.192201,0.514782,0.018114
204766,qw4OqKgP,20,black,NaN,-26.0,white_win,0.312935,0.185759,0.501307,0.019283
162608,oRD7N98j,173,white,NaN,-268.0,white_win,0.286083,0.215309,0.498608,0.020736
204765,qw4OqKgP,19,white,NaN,-38.0,white_win,0.294514,0.186188,0.519298,0.024358
196088,zLh06Egr,6,black,NaN,-14.0,black_win,0.286678,0.193419,0.519903,0.024838
186759,3f0tKeFx,66,black,NaN,150.0,black_win,0.278820,0.212047,0.509133,0.026022
162616,oRD7N98j,181,white,NaN,-278.0,white_win,0.283582,0.222170,0.494248,0.028181
204755,qw4OqKgP,9,white,NaN,-35.0,white_win,0.317073,0.173440,0.509487,0.032969
204763,qw4OqKgP,17,white,NaN,-25.0,white_win,0.326520,0.192836,0.480644,0.033605
204775,qw4OqKgP,29,white,NaN,-46.0,white_win,0.296339,0.177542,0.526119,0.034641
